# Notebook 2: Camera Adversarial Attacks

Demonstrate all 8 camera attack types on IR and EO cameras.

**Attacks:** FGSM, PGD, BIM, C&W, Universal, Backdoor, Physical, EOT

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from helpers import load_scenario, get_all_detections, CameraAttackerDF
from attacks.camera_attacks import AttackType

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 8)

## 2.1 Load Data

In [ ]:
SCENARIO = 'scenario2'
loader = load_scenario(SCENARIO)
detections = get_all_detections(loader)

ir_detections = detections[3]
eo_detections = detections[4]

print('IR Camera:', len(ir_detections), 'detections')
print('EO Camera:', len(eo_detections), 'detections')

## 2.2 Initialize Attacker

In [ ]:
attacker = CameraAttackerDF(epsilon=0.05, num_steps=10)
print('Available attacks:', [a.name for a in AttackType])

## 2.3 Run Individual Attacks on IR Camera

In [ ]:
attacks_to_demo = [
    AttackType.FGSM,
    AttackType.PGD,
    AttackType.BIM,
    AttackType.CW,
    AttackType.UNIVERSAL,
    AttackType.BACKDOOR,
    AttackType.PHYSICAL,
    AttackType.EOT
]

results = {}
for attack_type in attacks_to_demo:
    attacked = attacker.attack_detections(ir_detections.copy(), attack_type, sensor_id=3)
    results[attack_type.name] = attacked
    print(attack_type.name + ':', len(attacked), 'detections')

## 2.4 Visualize Attack Impact (Bearings)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for idx, (attack_name, attacked_df) in enumerate(results.items()):
    ax = axes[idx]
    benign_bearings = ir_detections['bearing'].dropna().values
    attacked_bearings = attacked_df['bearing'].dropna().values
    
    ax.hist(benign_bearings, bins=50, alpha=0.5, label='Benign', density=True)
    ax.hist(attacked_bearings, bins=50, alpha=0.5, label='Attacked', density=True)
    
    shift = np.mean(np.abs(attacked_bearings[:len(benign_bearings)] - benign_bearings[:len(attacked_bearings)]))
    
    ax.set_title(attack_name + '\nMean shift: ' + str(round(shift, 4)) + ' rad')
    ax.set_xlabel('Bearing (rad)')
    ax.set_ylabel('Density')
    ax.legend()
    ax.grid(True)

plt.suptitle('Camera Attack Impact on IR Camera Bearings', fontsize=14)
plt.tight_layout()
plt.show()

## 2.5 Spatial Impact Visualization

In [ ]:
attack_name = 'FGSM'
attacked_df = results[attack_name]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.scatter(ir_detections['x_piren'], ir_detections['y_piren'], s=5, alpha=0.5, c='blue')
ax1.set_title('Benign IR Camera Detections')
ax1.set_xlabel('East (m)')
ax1.set_ylabel('North (m)')
ax1.grid(True)
ax1.set_aspect('equal')

ax2.scatter(attacked_df['x_piren'], attacked_df['y_piren'], s=5, alpha=0.5, c='red')
ax2.set_title(attack_name + ' Attacked IR Camera Detections')
ax2.set_xlabel('East (m)')
ax2.set_ylabel('North (m)')
ax2.grid(True)
ax2.set_aspect('equal')

plt.tight_layout()
plt.show()

## 2.6 Epsilon Sweep

In [ ]:
epsilons = [0.01, 0.02, 0.05, 0.1, 0.2]
shifts = []

for eps in epsilons:
    att = CameraAttackerDF(epsilon=eps, num_steps=10)
    attacked = att.attack_detections(ir_detections.copy(), AttackType.FGSM, sensor_id=3)
    b = ir_detections['bearing'].dropna().values
    a = attacked['bearing'].dropna().values
    shift = np.mean(np.abs(a[:len(b)] - b[:len(a)]))
    shifts.append(shift)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epsilons, shifts, 'o-', linewidth=2, markersize=8)
ax.set_xlabel('Epsilon (rad)')
ax.set_ylabel('Mean Bearing Shift (rad)')
ax.set_title('FGSM Attack Strength vs Epsilon')
ax.grid(True)
plt.tight_layout()
plt.show()